In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as func

device = 'cuda' if torch.cuda.is_available() else 'cpu'

block_size = 8
batch_size = 4

In [11]:
with open("moby-dick.txt", "r", encoding='utf-8') as f:
    text = f.read()

vocab = sorted(set(text))
vocab_size = len(chars)

In [12]:
string_to_int = { ch:i for i,ch in enumerate(vocab) }
int_to_string = { i:ch for i,ch in enumerate(vocab) }

encode = lambda s: [string_to_int[c] for c in s ]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [6]:
n = int(0.8 * len(data))

train_data = data[:n]
valid_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')

print('inputs:')
print(x)

print('targets:')
print(y)

tensor([929147, 577417, 141378,  24490])
inputs:
tensor([[64, 61, 66, 59,  1, 68, 67, 61],
        [70, 53, 64,  8,  1, 72, 60, 57],
        [60, 70, 57, 57,  0, 71, 60, 61],
        [60, 57,  0, 74, 53, 71, 72,  1]], device='cuda:0')
targets:
tensor([[61, 66, 59,  1, 68, 67, 61, 66],
        [53, 64,  8,  1, 72, 60, 57,  1],
        [70, 57, 57,  0, 71, 60, 61, 68],
        [57,  0, 74, 53, 71, 72,  1, 53]], device='cuda:0')


In [ ]:
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        BATCH, TIME, CHANNEL = logits.shape

        if targets == None:
            loss = None
        else:
            logits = logits.view(BATCH * TIME, CHANNEL)
            targets = targets.view(BATCH * TIME)
            loss = func.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            
            logits = logits[:, -1, :]
            probs = func.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)

        return index

model = BigramLM(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
gen_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(gen_chars)